# 🫁 Pneumonia Detection — Training Notebook

**Instructions :**
1. `Runtime → Change runtime type → T4 GPU`
2. Lance les cellules dans l'ordre

> Tout est sur le GitHub du prof — un seul `git clone` suffit.

## 0 — Vérifier le GPU

In [ ]:
import torch
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
else:
    print('⚠️  Pas de GPU — va dans Runtime → Change runtime type → T4 GPU')

## 1 — Cloner le repo du prof

In [ ]:
import os
from pathlib import Path

# ── À modifier : URL du repo GitHub du prof ──────────────────────────────────
GITHUB_URL  = 'https://github.com/PROF_PSEUDO/PROF_REPO.git'   # <-- remplace ici
PROJECT_DIR = Path('/content/project')
# ─────────────────────────────────────────────────────────────────────────────

if not PROJECT_DIR.exists():
    print('Clonage en cours...')
    ret = os.system(f'git clone --depth 1 {GITHUB_URL} {PROJECT_DIR}')
    if ret != 0:
        raise RuntimeError('❌ git clone a échoué — vérifie l\'URL')
    print('✅ Repo cloné')
else:
    os.system(f'git -C {PROJECT_DIR} pull')
    print('✅ Repo mis à jour')

DATA_DIR = PROJECT_DIR / 'data'
for split in ['train', 'val', 'test_for_students']:
    d = DATA_DIR / split
    if d.exists():
        count = sum(1 for f in d.rglob('*') if f.is_file())
        print(f'  {split}: {count} fichiers')
    else:
        print(f'  ⚠️  {split}: dossier manquant')

## 2 — Installer les dépendances

In [ ]:
os.system(f'pip install -q -r {PROJECT_DIR}/requirements.txt')
print('✅ Dépendances installées')

## 3 — Définition des modèles

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))

import torch
import torch.nn as nn
import torch.nn.functional as F

# ══════════════════════════════════════════════
#  U-Net
# ══════════════════════════════════════════════

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2, 2)
    def forward(self, x):
        f = self.conv(x)
        return f, self.pool(f)

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))

class UNet(nn.Module):
    def __init__(self, in_channels=3, base_filters=64, dropout_rate=0.3):
        super().__init__()
        f = base_filters
        self.enc1 = EncoderBlock(in_channels, f)
        self.enc2 = EncoderBlock(f,   f*2)
        self.enc3 = EncoderBlock(f*2, f*4)
        self.enc4 = EncoderBlock(f*4, f*8)
        self.bottleneck = DoubleConv(f*8, f*16)
        self.dec4 = DecoderBlock(f*16, f*8)
        self.dec3 = DecoderBlock(f*8,  f*4)
        self.dec2 = DecoderBlock(f*4,  f*2)
        self.dec1 = DecoderBlock(f*2,  f)
        self.pool       = nn.AdaptiveAvgPool2d((1,1))
        self.dropout    = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(f, 1)
    def forward(self, x):
        s1,x = self.enc1(x); s2,x = self.enc2(x)
        s3,x = self.enc3(x); s4,x = self.enc4(x)
        x = self.dec4(self.bottleneck(x), s4)
        x = self.dec3(x, s3); x = self.dec2(x, s2); x = self.dec1(x, s1)
        return self.classifier(self.dropout(self.pool(x).flatten(1)))


# ══════════════════════════════════════════════
#  ResNet
# ══════════════════════════════════════════════

class ResidualBlock(nn.Module):
    expansion = 1
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x): return self.relu(self.main(x) + self.shortcut(x))

class ResNet(nn.Module):
    def __init__(self, in_channels=3, base_filters=64, dropout_rate=0.3):
        super().__init__()
        self.in_channels = base_filters
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base_filters, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(base_filters), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(base_filters,   2, stride=1)
        self.layer2 = self._make_layer(base_filters*2, 2, stride=2)
        self.layer3 = self._make_layer(base_filters*4, 2, stride=2)
        self.layer4 = self._make_layer(base_filters*8, 2, stride=2)
        self.pool       = nn.AdaptiveAvgPool2d((1,1))
        self.dropout    = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(base_filters*8, 1)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def _make_layer(self, out_ch, n_blocks, stride):
        layers = [ResidualBlock(self.in_channels, out_ch, stride)]
        self.in_channels = out_ch
        for _ in range(1, n_blocks): layers.append(ResidualBlock(out_ch, out_ch))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.classifier(self.dropout(self.pool(x).flatten(1)))


# ══════════════════════════════════════════════
#  Inception
# ══════════════════════════════════════════════

def conv_bn_relu(in_ch, out_ch, kernel, stride=1, padding=0):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=padding, bias=False),
        nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
    )

class InceptionBlock(nn.Module):
    def __init__(self, in_ch, b1, b2r, b2o, b3r, b3o, b4o):
        super().__init__()
        self.branch1 = conv_bn_relu(in_ch, b1, 1)
        self.branch2 = nn.Sequential(nn.Conv2d(in_ch,b2r,1),nn.ReLU(inplace=True),nn.Conv2d(b2r,b2o,3,padding=1),nn.ReLU(inplace=True))
        self.branch3 = nn.Sequential(nn.Conv2d(in_ch,b3r,1),nn.ReLU(inplace=True),nn.Conv2d(b3r,b3o,5,padding=2),nn.ReLU(inplace=True))
        self.branch4 = nn.Sequential(nn.MaxPool2d(3,stride=1,padding=1),nn.Conv2d(in_ch,b4o,1),nn.ReLU(inplace=True))
    def forward(self, x):
        return torch.cat([self.branch1(x),self.branch2(x),self.branch3(x),self.branch4(x)], dim=1)

class Inception(nn.Module):
    def __init__(self, in_channels=3, dropout_rate=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            conv_bn_relu(in_channels, 32, 3, stride=2, padding=1),
            conv_bn_relu(32, 64, 3, padding=1),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.block1 = InceptionBlock(64,  32,16,32,  8,16,16)
        self.pool1  = nn.MaxPool2d(3, stride=2, padding=1)
        self.block2 = InceptionBlock(96,  64,32,64, 16,32,32)
        self.pool2  = nn.MaxPool2d(3, stride=2, padding=1)
        self.block3 = InceptionBlock(192, 96,48,96, 24,48,48)
        self.block4 = InceptionBlock(288, 96,48,96, 24,48,48)
        self.pool       = nn.AdaptiveAvgPool2d((1,1))
        self.dropout    = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(288, 1)
    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(self.block1(x))
        x = self.pool2(self.block2(x))
        x = self.block4(self.block3(x))
        return self.classifier(self.dropout(self.pool(x).flatten(1)))


MODEL_REGISTRY = {'U-Net': UNet, 'ResNet': ResNet, 'Inception': Inception}

dummy = torch.randn(2, 3, 224, 224)
for name, cls in MODEL_REGISTRY.items():
    print(f'{name} → {cls()(dummy).shape}')  # attendu: torch.Size([2, 1])

## 4 — Transforms et DataLoaders

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def get_transforms(image_size, augment=False):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    if augment:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            normalize,
        ])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        normalize,
    ])

def get_dataloaders(image_size=224, batch_size=32):
    train_ds = datasets.ImageFolder(DATA_DIR / 'train', transform=get_transforms(image_size, augment=True))
    val_ds   = datasets.ImageFolder(DATA_DIR / 'val',   transform=get_transforms(image_size))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    print(f'Train: {len(train_ds)} images | Val: {len(val_ds)} images | Classes: {train_ds.classes}')
    return train_loader, val_loader, train_ds

## 5 — Boucle d'entraînement optimisée

Améliorations vs baseline :
- **AdamW** + weight decay → meilleure généralisation
- **Class weights automatiques** → gère le déséquilibre PNEUMONIA/NORMAL
- **Warmup + CosineAnnealingLR** → convergence plus stable
- **Gradient clipping** → évite les explosions de gradient
- **Label smoothing** → réduit l'overfitting

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

WEIGHTS_DIR = Path('/content/weights')
WEIGHTS_DIR.mkdir(exist_ok=True)


def compute_pos_weight(train_ds):
    """Calcule pos_weight = n_normal / n_pneumonia pour BCEWithLogitsLoss."""
    counts = np.bincount([label for _, label in train_ds.samples])
    pos_weight = torch.tensor([counts[0] / counts[1]], dtype=torch.float32).to(DEVICE)
    print(f'  Classes: {train_ds.classes} | counts: {counts} | pos_weight: {pos_weight.item():.3f}')
    return pos_weight


class LabelSmoothingBCE(nn.Module):
    """BCE with logits + label smoothing (smoothing=0.1 réduit l'overconfidence)."""
    def __init__(self, pos_weight=None, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    def forward(self, logits, targets):
        targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits, targets)


def train_one_epoch(model, loader, optimizer, criterion, clip_grad=1.0):
    model.train()
    total_loss, all_labels, all_probs = 0.0, [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip_grad)  # gradient clipping
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())
    avg_loss = total_loss / len(loader.dataset)
    accuracy = float(np.mean((np.array(all_probs) >= 0.5) == np.array(all_labels)))
    auc      = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5
    return avg_loss, accuracy, auc


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_labels, all_probs = 0.0, [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)
            logits = model(images)
            loss   = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            all_probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            all_labels.extend(labels.cpu().numpy().flatten())
    avg_loss = total_loss / len(loader.dataset)
    accuracy = float(np.mean((np.array(all_probs) >= 0.5) == np.array(all_labels)))
    auc      = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5
    return avg_loss, accuracy, auc


def train_model(model_name, lr=3e-4, epochs=30, batch_size=32, dropout=0.3,
                image_size=224, weight_decay=1e-4, warmup_epochs=3, label_smoothing=0.1):
    print(f'\n═══ {model_name} | lr={lr} epochs={epochs} bs={batch_size} wd={weight_decay} ═══')
    train_loader, val_loader, train_ds = get_dataloaders(image_size, batch_size)

    model     = MODEL_REGISTRY[model_name](in_channels=3, dropout_rate=dropout).to(DEVICE)
    pos_weight = compute_pos_weight(train_ds)

    optimizer  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion  = LabelSmoothingBCE(pos_weight=pos_weight, smoothing=label_smoothing)

    # Warmup linéaire sur `warmup_epochs`, puis cosine jusqu'à la fin
    warmup    = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine    = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])

    history = {k: [] for k in ['train_losses','val_losses','train_accs','val_accs','train_aucs','val_aucs']}
    best_val_auc = 0.0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, tr_auc = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, vl_auc = evaluate(model, val_loader, criterion)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        for k, v in zip(history, [tr_loss, vl_loss, tr_acc, vl_acc, tr_auc, vl_auc]):
            history[k].append(v)

        tag = ''
        if vl_auc > best_val_auc:
            best_val_auc = vl_auc
            torch.save(model.state_dict(), WEIGHTS_DIR / f"{model_name.replace(' ','_')}_best.pt")
            tag = '  ★ saved'

        print(f'Ep {epoch:02d}/{epochs} | lr={current_lr:.2e} | '
              f'Train loss={tr_loss:.4f} auc={tr_auc:.4f} | '
              f'Val   loss={vl_loss:.4f} auc={vl_auc:.4f}{tag}')

    print(f'\n✅ Meilleur Val AUC ({model_name}): {best_val_auc:.4f}')
    return model, history, best_val_auc

## 6 — Lancer l'entraînement

Change `MODEL_TO_TRAIN` : `'ResNet'` | `'U-Net'` | `'Inception'`

In [ ]:
# ── Hyperparamètres optimisés ─────────────────────────────────────────────────
MODEL_TO_TRAIN  = 'ResNet'   # 'ResNet' | 'U-Net' | 'Inception'
LR              = 3e-4       # AdamW fonctionne mieux avec un lr plus faible
EPOCHS          = 30
BATCH_SIZE      = 32
DROPOUT         = 0.3
IMAGE_SIZE      = 224
WEIGHT_DECAY    = 1e-4       # régularisation L2
WARMUP_EPOCHS   = 3          # montée en lr progressive
LABEL_SMOOTHING = 0.1        # réduit l'overconfidence
# ─────────────────────────────────────────────────────────────────────────────

model, history, best_auc = train_model(
    MODEL_TO_TRAIN, lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE,
    dropout=DROPOUT, image_size=IMAGE_SIZE, weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS, label_smoothing=LABEL_SMOOTHING,
)

## 7 — Courbes d'entraînement

In [ ]:
import matplotlib.pyplot as plt

def plot_curves(history, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ep = range(1, len(history['train_losses']) + 1)
    for ax, tk, vk, title in [
        (axes[0], 'train_losses', 'val_losses', 'Loss'),
        (axes[1], 'train_accs',  'val_accs',   'Accuracy'),
        (axes[2], 'train_aucs',  'val_aucs',   'AUC'),
    ]:
        ax.plot(ep, history[tk], 'b-', label='train')
        ax.plot(ep, history[vk], color='orange', linestyle='--', label='val')
        ax.set_title(title); ax.set_xlabel('Epoch'); ax.grid(True); ax.legend()
    axes[2].axhline(0.95, color='red', linestyle='--', label='target'); axes[2].legend()
    fig.suptitle(f'{model_name} — Training curves', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()

plot_curves(history, MODEL_TO_TRAIN)

## 8 — Prédictions avec TTA + CSV de soumission

**Test-Time Augmentation (TTA)** : on fait `N_TTA` passes sur chaque image avec des augmentations aléatoires,
puis on moyenne les probabilités. Gain AUC typique : **+0.5 à +1.5 points**.

In [ ]:
from PIL import Image
import pandas as pd
from google.colab import files

TEST_DIR = DATA_DIR / 'test_for_students'
N_TTA    = 8   # nombre de passes TTA (8 est un bon compromis vitesse/gain)

def get_tta_transforms(image_size):
    """Augmentations légères répétées à l'inférence."""
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        normalize,
    ])

def predict_with_tta(model_name, image_size=224, n_tta=N_TTA):
    weight_path = WEIGHTS_DIR / f"{model_name.replace(' ','_')}_best.pt"
    assert weight_path.exists(), f'Pas de poids pour {model_name} — entraîne-le d\'abord'

    m = MODEL_REGISTRY[model_name](in_channels=3).to(DEVICE)
    m.load_state_dict(torch.load(weight_path, map_location=DEVICE))
    m.eval()

    tf_base = get_transforms(image_size)          # 1 passe sans augmentation
    tf_tta  = get_tta_transforms(image_size)      # n_tta passes avec augmentation

    image_paths = (sorted(TEST_DIR.glob('*.jpeg')) +
                   sorted(TEST_DIR.glob('*.jpg'))  +
                   sorted(TEST_DIR.glob('*.png')))
    assert image_paths, f'Aucune image dans {TEST_DIR}'

    rows = []
    with torch.no_grad():
        for p in image_paths:
            img = Image.open(p).convert('RGB')

            # passe de base
            probs = [torch.sigmoid(m(tf_base(img).unsqueeze(0).to(DEVICE))).item()]

            # passes TTA
            for _ in range(n_tta):
                probs.append(torch.sigmoid(m(tf_tta(img).unsqueeze(0).to(DEVICE))).item())

            rows.append({'id': p.stem, 'prediction': round(float(np.mean(probs)), 6)})

    df = pd.DataFrame(rows)
    csv_name = f'submission_{model_name.replace(" ","_")}_tta.csv'
    df.to_csv(csv_name, index=False)
    print(f'✅ {len(df)} prédictions (TTA x{n_tta}) — téléchargement...')
    print(df['prediction'].describe().round(4))
    files.download(csv_name)
    return df

df_preds = predict_with_tta(MODEL_TO_TRAIN)

## (Bonus) — Entraîner les 3 modèles + ensemble

L'**ensemble** (moyenne des probabilités des 3 modèles) est souvent le meilleur résultat possible.

In [ ]:
# Décommente pour entraîner les 3 modèles et soumettre l'ensemble

# ── Entraîner les 3 ──────────────────────────────────────────────────────────
# all_results = {}
# for name in ['ResNet', 'U-Net', 'Inception']:
#     _, hist, auc = train_model(
#         name, lr=3e-4, epochs=30, batch_size=32, dropout=0.3,
#         image_size=224, weight_decay=1e-4, warmup_epochs=3, label_smoothing=0.1
#     )
#     all_results[name] = auc
#     plot_curves(hist, name)
#
# print('\nRésumé AUC :', all_results)
#
# ── Ensemble : moyenne des prédictions TTA des 3 modèles ─────────────────────
# dfs = {name: predict_with_tta(name) for name in ['ResNet', 'U-Net', 'Inception']}
#
# df_ensemble = dfs['ResNet'][['id']].copy()
# df_ensemble['prediction'] = (
#     dfs['ResNet']['prediction'] +
#     dfs['U-Net']['prediction']  +
#     dfs['Inception']['prediction']
# ) / 3
# df_ensemble['prediction'] = df_ensemble['prediction'].round(6)
#
# df_ensemble.to_csv('submission_ensemble.csv', index=False)
# files.download('submission_ensemble.csv')
# print('🏆 Ensemble soumis !')